# 02 · Describe synthetic retail visits and channel activity

This stage examines invented behavior without interpreting it as market demand or a causal relationship. **Every numerical result is SYNTHETIC:** illustrative results on independently generated synthetic data; not findings from the ADK consumer survey.

This notebook generates its own fixture. It can run independently after package installation. The preceding [data-understanding stage](01_data_understanding.ipynb) explains the schema and validation boundary.

In [ ]:
import pandas as pd
from IPython.display import display

from retail_case.schema import BEHAVIOR_COLUMNS, SYNTHETIC_NOTICE
from retail_case.synthetic import generate_data

df = generate_data(n=720, seed=2026, missing_rate=0.03)


def show_synthetic_table(title, table):
    print(f"SYNTHETIC — {title}")
    print(SYNTHETIC_NOTICE)
    display(table)

In [ ]:
from retail_case.validation import validate_data

validate_data(df)
visits = df[list(BEHAVIOR_COLUMNS)]
show_synthetic_table(
    "Visit-count descriptions; missing values excluded by column", visits.describe().T
)

paired = visits.dropna().copy()
paired["store_band"] = paired["monthly_store_visits"].map(
    lambda value: "at most 2 / month" if value <= 2 else "above 2 / month"
)
paired["other_band"] = paired["weekly_other_channel_visits"].map(
    lambda value: "at most 1 / week" if value <= 1 else "above 1 / week"
)
cross_channel = pd.crosstab(paired["store_band"], paired["other_band"], margins=True)
show_synthetic_table("Illustrative threshold bands; complete behavior pairs only", cross_channel)
print(
    f"SYNTHETIC paired denominator: {len(paired)}; "
    f"rows omitted for missing behavior: {len(df) - len(paired)}"
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
units = ("Store visits / month", "Other-channel visits / week")
for axis, column, unit in zip(axes, BEHAVIOR_COLUMNS, units, strict=True):
    values = df[column].dropna().sort_values().to_numpy()
    axis.step(values, (pd.Series(range(1, len(values) + 1)) / len(values)).to_numpy(), where="post")
    axis.set(title=f"SYNTHETIC: {column}", xlabel=unit, ylabel="Empirical cumulative share")
    axis.grid(alpha=0.2)
fig.suptitle("SYNTHETIC behavior distributions; missing values omitted")
fig.tight_layout()
plt.show()

## Interpretation limits

**Observation:** these tables and ECDFs describe this single generator run. **Assumption:** the threshold bands offer a readable exploratory view; they are not validated customer categories. **Unknown:** whether other-channel activity competes with or complements store visits. The two counts use different time periods, so their raw magnitudes are not directly comparable.

There is no population weighting, market denominator, causal identification, or real commercial outcome. [03 · Opportunity exploration](03_opportunity.ipynb) uses these descriptions to expose conditional hypotheses and evidence gaps.